In [11]:
import os

In [12]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender\\research'

In [13]:
os.chdir("../")

In [14]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender'

In [4]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    vectorizer_path: Path

In [5]:
from Book_recommender.constant import *
from Book_recommender.utils.common import read_yaml, create_directories

In [ ]:
class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            vectorizer_path= config.vectorizer_path
        )

        return data_transformation_config

In [7]:
from Book_recommender.logging.log import logger
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [19]:
#Component
class DataTransformation:
    def __init__(self, config):
        self.config = config
        self.vectorizer = None

    def initialize_vectorizer(self):
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=10000,
            ngram_range=(1, 2),
            min_df=2
        )

    def fit_transform(self, data):
        return self.vectorizer.fit_transform(
            data["combined_features"]
        )

    def save(self, path):
        # save vectorizer
        joblib.dump(self.vectorizer, path)
        logger.info(f"Vectorizer saved to {path}")

        df = pd.read_csv(self.config.data_path)

        # save tfidf matrix separately
        tfidf_matrix = self.vectorizer.transform(df["combined_features"])
        joblib.dump(tfidf_matrix, "artifacts/data_transformation/tfidf_matrix.pkl")
        logger.info("TF-IDF matrix saved")

In [20]:
try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()
    
    data = pd.read_csv("artifacts/data_preprocessing/processed_data.csv")  # load your data
    
    data_processing = DataTransformation(config=data_transformation_config)
    data_processing.initialize_vectorizer()
    data_processing.fit_transform(data)
    data_processing.save(data_transformation_config.vectorizer_path)
    
except Exception as e:
    raise e

[2026-06-02 20:11:30,014: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-06-02 20:11:30,016: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-06-02 20:11:30,019: INFO: common: created directory at artifacts]
[2026-06-02 20:11:30,020: INFO: common: created directory at artifacts/data_transformation]
[2026-06-02 20:11:30,828: INFO: 388565315: Vectorizer saved to artifacts/data_transformation/tfidf.pkl]
[2026-06-02 20:11:31,157: INFO: 388565315: TF-IDF matrix saved]
